# Cố định split V5 sau khi làm sạch dữ liệu

Không train/resume, không xóa file và không sao chép lại video. Không cần GPU. Notebook giữ membership của split cũ bằng cách dựng lại danh sách ban đầu từ bản sạch cộng đúng 4 đường dẫn đã loại. Chỉ dùng nếu không đổi tên hoặc xóa thêm file. Seed=42, val_ratio=0.2, train limit=2000, controller limit=400 như run đã gửi.

Các folder được tạo là liên kết dùng trong session. Giữ dataset sạch và file split_manifest.json để tái tạo sau khi đổi session.


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/proxy_v3")
CLEAN = Path(
    "/kaggle/working/cleaned_final/"
    "kinetics400_5per/kinetics400_5per/train"
)
# Nếu đã đổi session, CLEAN có thể trỏ tới dataset sạch trong Input.
# Bước này CHỈ ĐỌC CLEAN, không xóa gì.
SPLIT_ROOT = Path("/kaggle/working/v5_fixed_split")

assert CLEAN.is_dir(), f"Không tìm thấy bản sạch: {CLEAN}"
assert SPLIT_ROOT.resolve().is_relative_to(Path("/kaggle/working").resolve())
assert SPLIT_ROOT.resolve() != Path("/kaggle/working").resolve()

if not PROJECT.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/munnn01/proxy_v3.git",
        str(PROJECT),
    ], check=True)

assert (PROJECT / "preprocessing/data.py").is_file()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("CLEAN:", CLEAN)
print("SPLIT_ROOT:", SPLIT_ROOT)


In [ ]:
import json
import pandas as pd
from torchvision.models.video import R3D_18_Weights
from preprocessing.data import (
    VIDEO_SUFFIXES,
    normalize_label,
    stratified_split_indices,
    stratified_limit_indices,
)

SEED = 42
VAL_RATIO = 0.2
TRAIN_LIMIT = 2000
CONTROLLER_LIMIT = 400

REMOVED = {
    "swing dancing/tOeStguoPog.mp4",
    "playing paintball/DuSvOhH8JZY.mp4",
    "roller skating/Yyx1e918PdQ_raw.f251.webm",
    "squat/PV9sURwufw4_raw.f251.webm",
}

clean_paths = {
    p.relative_to(CLEAN).as_posix()
    for p in CLEAN.rglob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES
}
assert len(clean_paths) == 10801, (
    f"Đang có {len(clean_paths)} file, khác báo cáo 10801. "
    "Dừng để kiểm tra inventory trước khi dựng split."
)
assert not (clean_paths & REMOVED), (
    f"Bản sạch vẫn còn file đã loại: {clean_paths & REMOVED}"
)

# Dựng danh sách TRƯỚC khi xóa, rồi mới chạy thuật toán split cũ.
original_paths = sorted(clean_paths | REMOVED)
categories = list(R3D_18_Weights.DEFAULT.meta["categories"])
lookup = {normalize_label(name): i for i, name in enumerate(categories)}

samples = []
ignored = []
for relative_path in original_paths:
    path = Path(relative_path)
    label = lookup.get(normalize_label(path.parts[0]))
    if len(path.parts) < 2 or label is None:
        ignored.append(relative_path)
        continue
    samples.append((path, label))

train_pool_ids, full_val_ids = stratified_split_indices(
    samples, VAL_RATIO, SEED
)
train_ids = stratified_limit_indices(
    samples, train_pool_ids, TRAIN_LIMIT, SEED + 101
)
controller_ids = stratified_limit_indices(
    samples, full_val_ids, CONTROLLER_LIMIT, SEED + 202
)
assert len(full_val_ids) == 2155, (
    f"Full validation tái tạo được {len(full_val_ids)} video, "
    "khác run cũ 2155. Dừng, không tự thay đổi split."
)
assert len(train_ids) == 2000 and len(controller_ids) == 400

original_groups = {
    "train_pool": [samples[i][0].as_posix() for i in train_pool_ids],
    "train": [samples[i][0].as_posix() for i in train_ids],
    "controller": [samples[i][0].as_posix() for i in controller_ids],
    "validation_full": [samples[i][0].as_posix() for i in full_val_ids],
}
groups = {
    name: [p for p in paths if p in clean_paths]
    for name, paths in original_groups.items()
}

# Chỉ lọc file đã loại, KHÔNG lấy video mới để bù đủ 2000/400.
assert set(groups["train_pool"]).isdisjoint(groups["validation_full"])
assert set(groups["train"]) <= set(groups["train_pool"])
assert set(groups["controller"]) <= set(groups["validation_full"])
assert set(groups["train_pool"]) | set(groups["validation_full"]) == (
    {p.as_posix() for p, _ in samples} & clean_paths
)

manifest = {
    "method": "original_inventory_reconstructed_from_clean_plus_four_removed",
    "seed": SEED,
    "val_ratio": VAL_RATIO,
    "train_limit_before_filter": TRAIN_LIMIT,
    "controller_limit_before_filter": CONTROLLER_LIMIT,
    "clean_root": str(CLEAN.resolve()),
    "original_video_count": len(original_paths),
    "clean_video_count": len(clean_paths),
    "removed": sorted(REMOVED),
    "ignored_by_category_mapping": ignored,
    "original_groups": original_groups,
    "groups": groups,
}
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = SPLIT_ROOT / "split_manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text(encoding="utf-8")) == manifest, (
        "Thư mục split này đã chứa một cấu hình khác. "
        "Đổi SPLIT_ROOT, không ghi đè split cũ."
    )

# Không nhân đôi video: chỉ tạo liên kết đến bản sạch.
class_folders = [p.name for p in CLEAN.iterdir() if p.is_dir()]
for role in ("train", "controller", "validation_full"):
    role_root = SPLIT_ROOT / role
    role_root.mkdir(exist_ok=True)

    for class_name in class_folders:
        (role_root / class_name).mkdir(exist_ok=True)

    expected = set(groups[role])
    existing = {
        p.relative_to(role_root).as_posix()
        for p in role_root.rglob("*")
        if p.is_file() or p.is_symlink()
    }
    assert existing <= expected, f"Có file ngoài manifest trong {role}"

    for relative_path in groups[role]:
        source = (CLEAN / relative_path).resolve(strict=True)
        target = role_root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_symlink():
            assert target.resolve(strict=True) == source
        else:
            assert not target.exists(), f"Không ghi đè: {target}"
            target.symlink_to(source)

manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary = pd.DataFrame([
    {
        "role": role,
        "before_cleaning": len(original_groups[role]),
        "after_cleaning": len(groups[role]),
        "removed_paths": [
            p for p in original_groups[role] if p not in clean_paths
        ],
    }
    for role in original_groups
])
print(summary.to_string(index=False))
print("\nSố file không khớp category của analyzer:", len(ignored))
print("Manifest:", manifest_path)


In [ ]:
from preprocessing.data import VideoFolderDataset

TRAIN_DIR = SPLIT_ROOT / "train"
CONTROLLER_DIR = SPLIT_ROOT / "controller"
FULL_VAL_DIR = SPLIT_ROOT / "validation_full"

# Chỉ kiểm tra inventory; không đọc/giải mã video và không train.
train_dataset = VideoFolderDataset(TRAIN_DIR, categories, train=True)
controller_dataset = VideoFolderDataset(CONTROLLER_DIR, categories, train=False)
full_val_dataset = VideoFolderDataset(FULL_VAL_DIR, categories, train=False)

def resolved_paths(dataset):
    return {str(path.resolve(strict=True)) for path, _ in dataset.samples}

train_paths = resolved_paths(train_dataset)
controller_paths = resolved_paths(controller_dataset)
full_val_paths = resolved_paths(full_val_dataset)

assert len(train_dataset) == len(groups["train"])
assert len(controller_dataset) == len(groups["controller"])
assert len(full_val_dataset) == len(groups["validation_full"])
assert train_paths.isdisjoint(full_val_paths)
assert controller_paths <= full_val_paths

print("PASS: training và full validation không trùng file nguồn.")
print("PASS: controller nằm trong validation, không nằm trong training.")
print("TRAIN_DIR:", TRAIN_DIR)
print("CONTROLLER_DIR:", CONTROLLER_DIR)
print("FULL_VAL_DIR:", FULL_VAL_DIR)
print("Không có bước training nào được chạy.")
print("Chưa resume V5 hoặc tăng ngưỡng proxy guard.")


## Sau khi chạy

Gửi bảng số lượng ở Cell 2 và ba dòng đường dẫn ở Cell 3. Không quét hoặc sao chép lại video.

Việc tách folder nguồn train/validation tránh đường retry của loader hiện tại lấy file validation, **nếu dùng đúng các đường dẫn này ở bước sau**. Notebook không sửa train.py và không làm proxy chính xác hơn. Full validation này là split cũ đã dùng để phân tích, không phải final holdout mới.

Các kết quả cũ vẫn có nguy cơ đã bị ảnh hưởng bởi retry trước đây; dựng lại split không xóa được nguy cơ đó khỏi checkpoint cũ.
